# Chapter 17: Limited Dependent Variable Models and Sample Selection Corrections

Topics: binary responses (LPM, Logit, Probit), count data (Poisson), Tobit model, censored/truncated regression, sample selection.

## Step 1 – Setup: Working Directory & Output Folder

Run this cell first to fix the working directory and create the `output/` folder.

In [ ]:
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
import wooldridge as woo
import matplotlib.pyplot as plt
import os

output_dir = "output"
os.makedirs(output_dir, exist_ok=True)
df = woo.dataWoo('mroz')

## 17.1.1 Linear Probability Model

## Step 2 – Binary Dependent Variable Models

When $y_i \in \{0, 1\}$, we model $P(y_i=1|\mathbf{x}_i)$.

**Linear Probability Model (LPM):**
$$P(y=1|\mathbf{x}) = \mathbf{x}'\boldsymbol{\beta}$$

Drawback: predicted probabilities can be outside $[0,1]$.

**Logit model** (logistic function $\Lambda$):
$$P(y=1|\mathbf{x}) = \Lambda(\mathbf{x}'\boldsymbol{\beta}) = \frac{e^{\mathbf{x}'\boldsymbol{\beta}}}{1+e^{\mathbf{x}'\boldsymbol{\beta}}}$$

**Probit model** (standard normal CDF $\Phi$):
$$P(y=1|\mathbf{x}) = \Phi(\mathbf{x}'\boldsymbol{\beta})$$

Both logit and probit are estimated by **Maximum Likelihood Estimation (MLE)**.

In [ ]:
lpm = smf.ols('inlf ~ nwifeinc + educ + exper + I(exper**2) + age + kidslt6 + kidsge6',
               data=df).fit()
print("Linear Probability Model:")
print(lpm.summary())
print(f"\nWarning: {(lpm.fittedvalues < 0).sum()} predictions < 0, "
      f"{(lpm.fittedvalues > 1).sum()} predictions > 1")

## 17.1.2 Logit and Probit

## Step 3 – Marginal Effects

Raw coefficients from logit/probit are **not directly interpretable** as marginal effects.

**Average Marginal Effect (AME):**
$$AME_j = \frac{1}{n}\sum_{i=1}^n \frac{\partial P(y_i=1|\mathbf{x}_i)}{\partial x_{ij}}$$

- **Logit AME:** $\frac{1}{n}\sum_i \hat{\beta}_j \cdot \hat{p}_i(1-\hat{p}_i)$ where $\hat{p}_i = \Lambda(\mathbf{x}_i'\hat{\boldsymbol{\beta}})$
- **Probit AME:** $\frac{1}{n}\sum_i \hat{\beta}_j \cdot \phi(\mathbf{x}_i'\hat{\boldsymbol{\beta}})$

AMEs from LPM, Logit and Probit are typically similar.

In [ ]:
logit  = smf.logit('inlf ~ nwifeinc + educ + exper + I(exper**2) + age + kidslt6',
                    data=df).fit()
probit = smf.probit('inlf ~ nwifeinc + educ + exper + I(exper**2) + age + kidslt6',
                     data=df).fit()
print("Logit Results:")
print(logit.summary())

## 17.1.5 Marginal Effects

## Step 4 – Count Data (Poisson) and Censored Data (Tobit)

**Poisson regression** for count outcomes ($y \in \{0,1,2,\ldots\}$):
$$E(y|\mathbf{x}) = \exp(\mathbf{x}'\boldsymbol{\beta})$$

**Incidence Rate Ratio (IRR):** $\exp(\hat{\beta}_j)$ — multiplicative factor change in the expected count per unit increase in $x_j$.

**Tobit model** for censored outcomes (e.g., $y \geq 0$):
$$y^* = \mathbf{x}'\boldsymbol{\beta} + u, \quad u \sim N(0,\sigma^2)$$
$$y = \max(0,\, y^*)$$

Expected observed value:
$$E(y|\mathbf{x}) = \Phi\!\left(\frac{\mathbf{x}'\boldsymbol{\beta}}{\sigma}\right)\mathbf{x}'\boldsymbol{\beta} + \sigma\,\phi\!\left(\frac{\mathbf{x}'\boldsymbol{\beta}}{\sigma}\right)$$

In [ ]:
mfx_logit  = logit.get_margeff()
mfx_probit = probit.get_margeff()

comp = pd.DataFrame({
    'LPM':    lpm.params.drop('Intercept'),
    'Logit AME':  mfx_logit.margeff,
    'Probit AME': mfx_probit.margeff
})
print("Average Marginal Effects Comparison:")
print(comp.round(4))

## 17.2 Count Data: Poisson Regression

In [ ]:
df_fert = woo.dataWoo('fertil1')
poisson = smf.poisson('kids ~ educ + age + agesq + black + east + northcen + west + farm',
                       data=df_fert).fit()
print("Poisson Results:")
print(poisson.summary())

# Incidence rate ratios
irr = np.exp(poisson.params)
print("\nIncidence Rate Ratios:")
print(irr.round(4))

## 17.3 Tobit Model

In [ ]:
# Hours worked (censored at 0 for non-participants)
# Standard Tobit via statsmodels (OLS on uncensored subsample as approximation)
tobit_ols = smf.ols('hours ~ nwifeinc + educ + exper + I(exper**2) + age + kidslt6',
                     data=df[df['inlf']==1]).fit()
print("OLS on uncensored subsample (upper bound of Tobit):")
print(tobit_ols.summary())